# 05 — Fusion Multimodale : Fundus + OCT

**Objectif :** Combiner les probabilités OOF (Out-Of-Fold) des modèles Fundus et OCT pour construire un classifieur de fusion.

## Inputs requis
| Fichier | Source | Description |
|---------|--------|-------------|
| `fundus_probabilities.npy` | Notebook 03 output | Probas OOF Fundus — shape=(48,) |
| `oct_probabilities.npy` | Notebook 04 output | Probas OOF OCT — shape=(48,) |
| `fundus_labels.npy` ou `oct_labels.npy` | Notebook 03 ou 04 | Labels ground truth — shape=(48,) |

## Approches comparées
| Méthode | Description |
|---------|-------------|
| **Moyenne simple** | (P_fundus + P_oct) / 2 |
| **Moyenne pondérée** | α·P_fundus + (1-α)·P_oct, α optimisé sur grille |
| **Régression Logistique** | sklearn LogisticRegression sur [P_fundus, P_oct] |
| **MLP** | PyTorch : Linear(2→16)→ReLU→Dropout→Linear(16→8)→ReLU→Linear(8→1) |

## Architecture MLP
```
Input : [P_fundus, P_oct]  (dim=2)
  Linear(2 → 16) + ReLU + Dropout(0.3)
  Linear(16 → 8) + ReLU
  Linear(8 → 1)   → logit → BCEWithLogitsLoss
```

## ⚠️ Configuration Kaggle
Les outputs des notebooks 03 et 04 doivent être attachés comme datasets input :
- Kaggle → Your Work → [notebook 03] → Output → Add as Dataset input
- Kaggle → Your Work → [notebook 04] → Output → Add as Dataset input
- Les fichiers `.npy` seront disponibles sous `/kaggle/input/`

## Section 1 : Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve
)
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444',
    'axes.labelcolor': '#e0e0e0',
    'axes.titlecolor': '#ffffff',
    'xtick.color': '#aaaaaa',
    'ytick.color': '#aaaaaa',
    'text.color': '#e0e0e0',
    'grid.color': '#333355',
    'grid.linestyle': '--',
    'grid.alpha': 0.4,
    'font.size': 11,
})

PALETTE = {
    'fundus'   : '#e74c3c',
    'oct'      : '#2ecc71',
    'avg'      : '#3498db',
    'wavg'     : '#f39c12',
    'lr'       : '#9b59b6',
    'mlp'      : '#1abc9c',
    'baseline' : '#7f8c8d',
}

OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Imports OK')
print(f'   PyTorch  : {torch.__version__}')
print(f'   Device   : {DEVICE}')

## Section 2 : Chargement des Probabilités OOF

> **Kaggle :** Attacher les outputs des notebooks 03 et 04 comme datasets input.  
> Les fichiers `.npy` seront disponibles sous `/kaggle/input/<dataset-name>/outputs/`

In [ ]:
# ── Détection automatique des chemins ─────────────────────────────────────
# Cherche les fichiers .npy dans tous les inputs Kaggle disponibles

def find_npy(filename, search_roots=None):
    """Cherche un fichier .npy dans les chemins Kaggle habituels."""
    if search_roots is None:
        search_roots = [
            '/kaggle/input',
            '/kaggle/working/outputs',
            '/kaggle/working',
        ]
    for root in search_roots:
        for dirpath, _, files in os.walk(root):
            if filename in files:
                return os.path.join(dirpath, filename)
    return None

# ── Localisation des fichiers ──────────────────────────────────────────────
path_fundus_probs  = find_npy('fundus_probabilities.npy')
path_oct_probs     = find_npy('oct_probabilities.npy')
path_labels        = find_npy('fundus_labels.npy') or find_npy('oct_labels.npy')

print('🔍 Recherche des fichiers OOF :')
print(f'   fundus_probabilities.npy : {path_fundus_probs or "❌ NON TROUVÉ"}')
print(f'   oct_probabilities.npy    : {path_oct_probs or "❌ NON TROUVÉ"}')
print(f'   labels                   : {path_labels or "❌ NON TROUVÉ"}')

assert path_fundus_probs, "❌ fundus_probabilities.npy introuvable — attacher l'output du notebook 03 comme dataset Kaggle"
assert path_oct_probs,    "❌ oct_probabilities.npy introuvable — attacher l'output du notebook 04 comme dataset Kaggle"
assert path_labels,       "❌ labels.npy introuvable"

# ── Chargement ────────────────────────────────────────────────────────────
p_fundus = np.load(path_fundus_probs)   # shape=(48,)
p_oct    = np.load(path_oct_probs)      # shape=(48,)
y_true   = np.load(path_labels).astype(int)  # shape=(48,)

print(f'\n✅ Données chargées :')
print(f'   p_fundus shape : {p_fundus.shape}  min={p_fundus.min():.3f}  max={p_fundus.max():.3f}  mean={p_fundus.mean():.3f}')
print(f'   p_oct    shape : {p_oct.shape}  min={p_oct.min():.3f}  max={p_oct.max():.3f}  mean={p_oct.mean():.3f}')
print(f'   y_true   shape : {y_true.shape}  Glaucome={y_true.sum()}  Normal={(y_true==0).sum()}')

# Vérification cohérence
assert len(p_fundus) == len(p_oct) == len(y_true), "❌ Tailles incohérentes !"
print('\n✅ Cohérence vérifiée — toutes les arrays ont la même taille')

## Section 3 : Analyse des Probabilités OOF

In [ ]:
# ── AUC individuels pour référence ────────────────────────────────────────
auc_fundus = roc_auc_score(y_true, p_fundus)
auc_oct    = roc_auc_score(y_true, p_oct)

print('📊 AUC individuels (OOF global) :')
print(f'   Fundus AUC : {auc_fundus:.4f}')
print(f'   OCT    AUC : {auc_oct:.4f}')

# Corrélation entre les deux modèles
corr = np.corrcoef(p_fundus, p_oct)[0, 1]
print(f'   Corrélation Fundus/OCT : {corr:.4f}  ← plus faible = meilleure complémentarité')

# ── Scatter plot des probabilités ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Analyse des Probabilités OOF — Fundus vs OCT', fontsize=13, fontweight='bold')

# Scatter
ax = axes[0]
colors = [PALETTE['fundus'] if y == 1 else PALETTE['oct'] for y in y_true]
ax.scatter(p_fundus, p_oct, c=colors, alpha=0.75, s=80, edgecolors='white', linewidth=0.5)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(0.5, color='white', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xlabel('P_fundus'); ax.set_ylabel('P_oct')
ax.set_title(f'Scatter (corr={corr:.2f})')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=PALETTE['fundus'], label='Glaucome'),
                   Patch(facecolor=PALETTE['oct'], label='Normal')]
ax.legend(handles=legend_elements, fontsize=9)
ax.grid(True, alpha=0.3)

# Distribution Fundus
ax = axes[1]
ax.hist(p_fundus[y_true == 0], bins=12, alpha=0.7, color=PALETTE['oct'],    label='Normal',   density=True)
ax.hist(p_fundus[y_true == 1], bins=12, alpha=0.7, color=PALETTE['fundus'], label='Glaucome', density=True)
ax.axvline(0.5, color='white', linestyle='--', linewidth=1.5)
ax.set_title(f'Distribution P_fundus\n(AUC={auc_fundus:.3f})')
ax.set_xlabel('Probabilité'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Distribution OCT
ax = axes[2]
ax.hist(p_oct[y_true == 0], bins=12, alpha=0.7, color=PALETTE['oct'],    label='Normal',   density=True)
ax.hist(p_oct[y_true == 1], bins=12, alpha=0.7, color=PALETTE['fundus'], label='Glaucome', density=True)
ax.axvline(0.5, color='white', linestyle='--', linewidth=1.5)
ax.set_title(f'Distribution P_oct\n(AUC={auc_oct:.3f})')
ax.set_xlabel('Probabilité'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fusion_oof_analysis.png', bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✅ Analyse OOF sauvegardée')

## Section 4 : Méthode 1 — Moyenne Simple

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, name=''):
    """Calcule toutes les métriques à partir des probabilités."""
    y_pred = (y_prob >= threshold).astype(int)
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'name': name, 'accuracy': acc, 'f1': f1, 'roc_auc': auc,
        'sensitivity': sens, 'specificity': spec,
        'y_prob': y_prob, 'y_pred': y_pred, 'cm': cm
    }


# ── Moyenne simple ─────────────────────────────────────────────────────────
p_avg = (p_fundus + p_oct) / 2.0
results_avg = compute_metrics(y_true, p_avg, name='Moyenne simple')

print('📊 Moyenne simple : (P_fundus + P_oct) / 2')
print(f'   Accuracy    : {results_avg["accuracy"]*100:.2f}%')
print(f'   F1 Score    : {results_avg["f1"]:.4f}')
print(f'   ROC-AUC     : {results_avg["roc_auc"]:.4f}')
print(f'   Sensibilité : {results_avg["sensitivity"]*100:.2f}%')
print(f'   Spécificité : {results_avg["specificity"]*100:.2f}%')

## Section 5 : Méthode 2 — Moyenne Pondérée (Grid Search)

In [ ]:
# ── Grid search sur alpha ──────────────────────────────────────────────────
# p_fusion = alpha * p_fundus + (1 - alpha) * p_oct
# alpha optimisé pour maximiser ROC-AUC

alphas = np.linspace(0.0, 1.0, 101)  # 0.00, 0.01, ..., 1.00
aucs   = []

for alpha in alphas:
    p_fused = alpha * p_fundus + (1 - alpha) * p_oct
    aucs.append(roc_auc_score(y_true, p_fused))

best_alpha = alphas[np.argmax(aucs)]
best_auc   = max(aucs)

print(f'🔍 Grid search terminé :')
print(f'   Meilleur alpha : {best_alpha:.2f}  (α=0 → OCT pur, α=1 → Fundus pur)')
print(f'   AUC max        : {best_auc:.4f}')

p_wavg = best_alpha * p_fundus + (1 - best_alpha) * p_oct
results_wavg = compute_metrics(y_true, p_wavg, name=f'Moy. pondérée (α={best_alpha:.2f})')

print(f'\n📊 Moyenne pondérée optimale :')
print(f'   Accuracy    : {results_wavg["accuracy"]*100:.2f}%')
print(f'   F1 Score    : {results_wavg["f1"]:.4f}')
print(f'   ROC-AUC     : {results_wavg["roc_auc"]:.4f}')
print(f'   Sensibilité : {results_wavg["sensitivity"]*100:.2f}%')
print(f'   Spécificité : {results_wavg["specificity"]*100:.2f}%')

# Courbe AUC vs alpha
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(alphas, aucs, color=PALETTE['wavg'], linewidth=2)
ax.axvline(best_alpha, color='white', linestyle='--', linewidth=1.5,
           label=f'α optimal={best_alpha:.2f} (AUC={best_auc:.4f})')
ax.axvline(0.5, color=PALETTE['baseline'], linestyle=':', linewidth=1,
           label='α=0.5 (moyenne simple)')
ax.fill_between(alphas, aucs, alpha=0.15, color=PALETTE['wavg'])
ax.set_xlabel('Alpha (poids Fundus)'); ax.set_ylabel('ROC-AUC')
ax.set_title('Grid Search : AUC vs Alpha de pondération')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fusion_alpha_search.png', bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## Section 6 : Méthode 3 — Régression Logistique (Cross-Validated)

In [ ]:
# ── Features ──────────────────────────────────────────────────────────────
# X = [P_fundus, P_oct, P_fundus * P_oct, |P_fundus - P_oct|]
# Ajout de features d'interaction pour enrichir la représentation

X = np.column_stack([
    p_fundus,
    p_oct,
    p_fundus * p_oct,                  # interaction multiplicative
    np.abs(p_fundus - p_oct),          # divergence entre les modèles
])

print(f'Features X shape : {X.shape}  → [P_f, P_o, P_f*P_o, |P_f - P_o|]')

# ── Cross-val pour éviter l'overfitting (48 samples !) ────────────────────
# Utiliser cross_val_predict pour obtenir des probas OOF honnêtes
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_model = LogisticRegression(
    C=0.1,             # forte régularisation — dataset petit
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

# cross_val_predict en mode 'predict_proba' → probas OOF
p_lr_oof = cross_val_predict(
    lr_model, X, y_true,
    cv=skf,
    method='predict_proba'
)[:, 1]  # probabilité classe 1 (Glaucome)

results_lr = compute_metrics(y_true, p_lr_oof, name='Régression Logistique')

print(f'\n📊 Régression Logistique (cross-validated, C=0.1) :')
print(f'   Accuracy    : {results_lr["accuracy"]*100:.2f}%')
print(f'   F1 Score    : {results_lr["f1"]:.4f}')
print(f'   ROC-AUC     : {results_lr["roc_auc"]:.4f}')
print(f'   Sensibilité : {results_lr["sensitivity"]*100:.2f}%')
print(f'   Spécificité : {results_lr["specificity"]*100:.2f}%')

# Entraîner le modèle final sur tout le dataset pour inférence
lr_model.fit(X, y_true)
print(f'\n   Coefficients appris (features : P_f, P_o, P_f*P_o, |P_f-P_o|) :')
print(f'   {lr_model.coef_[0].round(3)}')

## Section 7 : Méthode 4 — MLP PyTorch (Cross-Validated)

In [ ]:
# ── Architecture MLP ──────────────────────────────────────────────────────
class FusionMLP(nn.Module):
    """
    MLP léger pour fusion multimodale.

    Input  : [P_fundus, P_oct]  — dim=2
    Output : logit scalaire → BCEWithLogitsLoss
    """
    def __init__(self, input_dim=2, hidden1=16, hidden2=8, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


# ── Entraînement d'un fold MLP ─────────────────────────────────────────────
def train_mlp_fold(X_train, y_train, X_val, y_val,
                   n_epochs=300, lr=1e-3, device=DEVICE):
    """
    Entraîne le FusionMLP sur un fold et retourne les probas sur X_val.
    """
    # Tenseurs
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)

    # pos_weight
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    pos_w = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)

    model     = FusionMLP(input_dim=X_train.shape[1]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_val_loss = float('inf')
    best_weights  = None
    patience_cnt  = 0
    patience      = 40

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_tr), y_tr)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # Validation (sans gradient)
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va),
                                 torch.tensor(y_val, dtype=torch.float32).to(device)).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            import copy
            best_weights  = copy.deepcopy(model.state_dict())
            patience_cnt  = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                break

    # Inférence sur val
    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        logits = model(X_va)
        probs  = torch.sigmoid(logits).cpu().numpy()

    return probs, model


# ── Input features pour MLP : [P_fundus, P_oct] ──────────────────────────
# Note : MLP simple avec dim=2 pour éviter l'overfitting sur 48 samples
X_mlp = np.column_stack([p_fundus, p_oct]).astype(np.float32)

print(f'✅ FusionMLP défini')
total_p = sum(p.numel() for p in FusionMLP().parameters())
print(f'   Paramètres totaux : {total_p}')
print(f'   Architecture      : Linear(2→16)→ReLU→Dropout(0.3)→Linear(16→8)→ReLU→Linear(8→1)')
print(f'   Input features    : [P_fundus, P_oct]  — dim=2')

In [ ]:
# ── Cross-validation MLP ──────────────────────────────────────────────────
p_mlp_oof = np.zeros(len(y_true))
mlp_fold_aucs = []

print('🚀 Entraînement MLP cross-validated (K=5) :')
print(f'   {"Fold":<6} {"Val indices":<20} {"Fold AUC"}')
print('   ' + '-' * 40)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_mlp, y_true)):
    X_tr, X_va = X_mlp[train_idx], X_mlp[val_idx]
    y_tr, y_va = y_true[train_idx], y_true[val_idx]

    probs, _ = train_mlp_fold(X_tr, y_tr, X_va, y_va, n_epochs=500, lr=5e-3)
    p_mlp_oof[val_idx] = probs

    fold_auc = roc_auc_score(y_va, probs)
    mlp_fold_aucs.append(fold_auc)
    print(f'   {fold_idx+1:<6} {str(val_idx[:4].tolist()) + "...":<20} {fold_auc:.4f}')

print(f'\n   AUC moyen folds : {np.mean(mlp_fold_aucs):.4f} ± {np.std(mlp_fold_aucs):.4f}')

results_mlp = compute_metrics(y_true, p_mlp_oof, name='MLP Fusion')

print(f'\n📊 MLP Fusion (cross-validated) :')
print(f'   Accuracy    : {results_mlp["accuracy"]*100:.2f}%')
print(f'   F1 Score    : {results_mlp["f1"]:.4f}')
print(f'   ROC-AUC     : {results_mlp["roc_auc"]:.4f}')
print(f'   Sensibilité : {results_mlp["sensitivity"]*100:.2f}%')
print(f'   Spécificité : {results_mlp["specificity"]*100:.2f}%')

## Section 8 : Tableau Comparatif Complet

In [ ]:
# ── Baselines mono-modèles ─────────────────────────────────────────────────
results_fundus = compute_metrics(y_true, p_fundus, name='Fundus seul')
results_oct    = compute_metrics(y_true, p_oct,    name='OCT seul')

# ── Tableau récapitulatif ─────────────────────────────────────────────────
all_results = [results_fundus, results_oct, results_avg, results_wavg, results_lr, results_mlp]

df_compare = pd.DataFrame([{
    'Méthode'     : r['name'],
    'Accuracy'    : f"{r['accuracy']*100:.2f}%",
    'F1 Score'    : f"{r['f1']:.4f}",
    'ROC-AUC'     : f"{r['roc_auc']:.4f}",
    'Sensibilité' : f"{r['sensitivity']*100:.2f}%",
    'Spécificité' : f"{r['specificity']*100:.2f}%",
} for r in all_results])

print('\n' + '='*75)
print('  📊 COMPARAISON COMPLÈTE — Toutes les Méthodes')
print('='*75)
print(df_compare.to_string(index=False))
print('='*75)

# Identifier la meilleure méthode par AUC
best_idx    = np.argmax([r['roc_auc'] for r in all_results])
best_result = all_results[best_idx]
print(f'\n🏆 Meilleure méthode : {best_result["name"]}')
print(f'   ROC-AUC = {best_result["roc_auc"]:.4f}')
print(f'   Gain vs Fundus seul : +{(best_result["roc_auc"] - results_fundus["roc_auc"]):.4f}')
print(f'   Gain vs OCT seul    : +{(best_result["roc_auc"] - results_oct["roc_auc"]):.4f}')

## Section 9 : Visualisation — Comparaison ROC + Barres

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('Comparaison Complète — Fusion Multimodale Fundus + OCT',
             fontsize=14, fontweight='bold')

method_colors = [
    PALETTE['fundus'],
    PALETTE['oct'],
    PALETTE['avg'],
    PALETTE['wavg'],
    PALETTE['lr'],
    PALETTE['mlp'],
]
names_short = ['Fundus', 'OCT', 'Moy.', 'W-Moy.', 'LogReg', 'MLP']

# ── ROC curves ────────────────────────────────────────────────────────────
ax_roc = fig.add_subplot(gs[0, :])
for r, color, name in zip(all_results, method_colors, names_short):
    fpr, tpr, _ = roc_curve(y_true, r['y_prob'])
    ls = '--' if 'seul' in r['name'] else '-'
    lw = 1.5 if 'seul' in r['name'] else 2.5
    ax_roc.plot(fpr, tpr, color=color, linewidth=lw, linestyle=ls,
                label=f'{name} (AUC={r["roc_auc"]:.3f})')
ax_roc.plot([0,1],[0,1], color='gray', linestyle=':', linewidth=1.2)
ax_roc.set_title('Courbes ROC — Toutes les Méthodes')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.legend(fontsize=9, loc='lower right')
ax_roc.grid(True, alpha=0.3)
ax_roc.set_xlim(0, 1); ax_roc.set_ylim(0, 1.02)

# ── Barres comparatives ───────────────────────────────────────────────────
metrics_to_plot = [
    ('accuracy',    'Accuracy',    gs[1, 0]),
    ('roc_auc',     'ROC-AUC',     gs[1, 1]),
    ('sensitivity', 'Sensibilité', gs[1, 2]),
]

for metric_key, metric_label, gs_pos in metrics_to_plot:
    ax = fig.add_subplot(gs_pos)
    vals = [r[metric_key] for r in all_results]
    bars = ax.bar(names_short, vals, color=method_colors, alpha=0.85,
                  edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')
    ax.set_title(metric_label, fontweight='bold')
    ax.set_ylim(0, min(max(vals) + 0.15, 1.02))
    ax.tick_params(axis='x', labelsize=8, rotation=15)
    ax.grid(True, alpha=0.3, axis='y')
    # Ligne de référence OCT seul
    ax.axhline(results_oct[metric_key], color=PALETTE['oct'],
               linestyle='--', linewidth=1, alpha=0.6, label='OCT baseline')

plt.savefig(f'{OUTPUT_DIR}/fusion_comparison.png', bbox_inches='tight',
            facecolor=fig.get_facecolor(), dpi=120)
plt.show()
print('✅ Comparaison sauvegardée → fusion_comparison.png')

## Section 10 : Matrices de Confusion des Méthodes de Fusion

In [ ]:
fusion_results = [results_avg, results_wavg, results_lr, results_mlp]
fusion_colors  = [PALETTE['avg'], PALETTE['wavg'], PALETTE['lr'], PALETTE['mlp']]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Matrices de Confusion — Méthodes de Fusion', fontsize=13, fontweight='bold')

for ax, r, color in zip(axes, fusion_results, fusion_colors):
    sns.heatmap(
        r['cm'], annot=True, fmt='d',
        cmap=sns.light_palette(color, as_cmap=True),
        xticklabels=['Normal', 'Glaucome'],
        yticklabels=['Normal', 'Glaucome'],
        ax=ax, linewidths=0.5,
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    ax.set_title(
        f"{r['name']}\nACC={r['accuracy']*100:.1f}%  AUC={r['roc_auc']:.3f}",
        fontsize=9
    )
    ax.set_xlabel('Prédit', fontsize=9)
    ax.set_ylabel('Réel', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fusion_confusion_matrices.png', bbox_inches='tight',
            facecolor=fig.get_facecolor(), dpi=120)
plt.show()
print('✅ Matrices de confusion sauvegardées')

## Section 11 : Sauvegarde du Meilleur Modèle de Fusion

In [ ]:
# ── Entraîner le MLP final sur TOUT le dataset ────────────────────────────
# (utilisé pour l'inférence dans le notebook 06 — Déploiement)

print('🔁 Entraînement du modèle de fusion final (tout le dataset)...')

_, mlp_final = train_mlp_fold(
    X_mlp, y_true,       # train = tout le dataset
    X_mlp, y_true,       # val   = idem (pas d'early stopping sur le final)
    n_epochs=500,
    lr=5e-3
)

# ── Sauvegarder les modèles de fusion ─────────────────────────────────────
import pickle

# MLP PyTorch
torch.save({
    'model_state_dict': mlp_final.state_dict(),
    'architecture'    : {'input_dim': 2, 'hidden1': 16, 'hidden2': 8, 'dropout': 0.3},
    'best_alpha'      : best_alpha,
    'metrics_oof'     : {
        'mlp_auc'  : results_mlp['roc_auc'],
        'lr_auc'   : results_lr['roc_auc'],
        'wavg_auc' : results_wavg['roc_auc'],
        'avg_auc'  : results_avg['roc_auc'],
    }
}, f'{OUTPUT_DIR}/fusion_mlp.pth')

# Régression logistique
with open(f'{OUTPUT_DIR}/fusion_logreg.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

# Probabilités de fusion OOF (meilleure méthode)
best_probs = best_result['y_prob']
np.save(f'{OUTPUT_DIR}/fusion_probabilities.npy', best_probs)
np.save(f'{OUTPUT_DIR}/fusion_labels.npy',        y_true)

# Config JSON (pour le notebook 06)
import json
config_fusion = {
    'best_method'      : best_result['name'],
    'best_alpha'       : float(best_alpha),
    'mlp_architecture' : {'input_dim': 2, 'hidden1': 16, 'hidden2': 8, 'dropout': 0.3},
    'oof_metrics'      : {
        'fundus'  : {'auc': float(results_fundus['roc_auc']), 'acc': float(results_fundus['accuracy'])},
        'oct'     : {'auc': float(results_oct['roc_auc']),    'acc': float(results_oct['accuracy'])},
        'avg'     : {'auc': float(results_avg['roc_auc']),    'acc': float(results_avg['accuracy'])},
        'wavg'    : {'auc': float(results_wavg['roc_auc']),   'acc': float(results_wavg['accuracy'])},
        'logreg'  : {'auc': float(results_lr['roc_auc']),     'acc': float(results_lr['accuracy'])},
        'mlp'     : {'auc': float(results_mlp['roc_auc']),    'acc': float(results_mlp['accuracy'])},
    }
}

with open(f'{OUTPUT_DIR}/fusion_config.json', 'w') as f:
    json.dump(config_fusion, f, indent=2)

print('\n💾 Fichiers sauvegardés :')
for fname in ['fusion_mlp.pth', 'fusion_logreg.pkl',
              'fusion_probabilities.npy', 'fusion_labels.npy',
              'fusion_config.json']:
    path = Path(f'{OUTPUT_DIR}/{fname}')
    print(f'   ✅  {fname}  ({path.stat().st_size // 1024} KB)' if path.exists() else f'   ❌  {fname}')

## Section 12 : Validation Finale & Résumé

In [ ]:
print('\n' + '='*60)
print('  📊 RÉSULTATS FINAUX — Fusion Multimodale')
print('='*60)

rows = [
    ('Fundus seul (baseline)',        results_fundus),
    ('OCT seul (baseline)',           results_oct),
    ('Moyenne simple',                results_avg),
    (f'Moy. pondérée (α={best_alpha:.2f})', results_wavg),
    ('Régression Logistique',         results_lr),
    ('MLP Fusion ★',                  results_mlp),
]

print(f'  {"Méthode":<35} {"ACC":>7} {"F1":>7} {"AUC":>7} {"Sens":>7} {"Spec":>7}')
print('  ' + '-'*72)
for label, r in rows:
    marker = ' ←' if r == best_result else ''
    print(f'  {label:<35} {r["accuracy"]*100:>6.1f}% {r["f1"]:>7.4f} {r["roc_auc"]:>7.4f} '
          f'{r["sensitivity"]*100:>6.1f}% {r["specificity"]*100:>6.1f}%{marker}')

print('\n' + '='*60)
print(f'  🏆 Meilleure méthode : {best_result["name"]}')
print(f'     AUC global OOF   = {best_result["roc_auc"]:.4f}')
print(f'     Gain vs OCT seul = +{best_result["roc_auc"] - results_oct["roc_auc"]:.4f}')
print('='*60)

# Assertions de validation
print('\n🔍 Assertions de validation :')
assertions = [
    (results_mlp['roc_auc'] >= results_oct['roc_auc'] - 0.05,
     'MLP AUC ≥ OCT AUC - 0.05  (fusion au moins proche du meilleur unimodal)'),
    (best_result['roc_auc'] > results_fundus['roc_auc'],
     'Meilleure fusion > Fundus seul'),
    (Path(f'{OUTPUT_DIR}/fusion_mlp.pth').exists(),
     'fusion_mlp.pth sauvegardé'),
    (Path(f'{OUTPUT_DIR}/fusion_logreg.pkl').exists(),
     'fusion_logreg.pkl sauvegardé'),
    (Path(f'{OUTPUT_DIR}/fusion_config.json').exists(),
     'fusion_config.json sauvegardé'),
    (Path(f'{OUTPUT_DIR}/fusion_comparison.png').exists(),
     'fusion_comparison.png sauvegardé'),
]

all_ok = True
for cond, msg in assertions:
    status = '✅' if cond else '⚠️ '
    print(f'   {status}  {msg}')
    if not cond:
        all_ok = False

print()
if all_ok:
    print('🎉 NOTEBOOK 05 TERMINÉ — Fusion multimodale complète !')
else:
    print('⚠️  Certaines assertions échouées — vérifier les résultats ci-dessus')

print()
print('📦 Fichiers produits pour notebook 06 (Déploiement) :')
print('   • fusion_mlp.pth         → modèle MLP final')
print('   • fusion_logreg.pkl      → régression logistique finale')
print('   • fusion_config.json     → meilleure méthode + alpha optimal')
print('   • fusion_mlp.pth + fundus_model_best.pth + oct_model_best.pth → pipeline complet')
print()
print('➡️  Prochain notebook : 06_deployment.ipynb')
print('   Endpoint : POST /predict → { glaucoma_probability, prediction, fundus_prob, oct_prob }')